# exp006 — Phase 1, step 1: training-speed benchmark of the official ink_9um recipe (Kaggle T4 x2)

**Settings:** Accelerator **GPU T4 x2**, Internet **on** (turn it back on after enabling the GPU), notebook **private**.
Config: `configs/exp006_train_benchmark.yaml`. Downloads into `/kaggle/working` (discarded with the session): ~0.5 GB of sparse
volume chunks (S3) + ~1 MB of labels (HF). No checkpoint download. Expected duration 30-60 min. Numbers only, no images.
This notebook **trains small throwaway models** for 150-300 steps to measure speed; their quality is irrelevant.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
echo "cpus: $(nproc)"; free -g | sed -n 2p; df -h /kaggle/working | tail -1; python --version
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /kaggle/working/harness
git -C /kaggle/working/harness log -1 --format='harness %H'

In [ ]:
import os, yaml, pathlib
H = pathlib.Path("/kaggle/working/harness")
EXP = "configs/exp006_train_benchmark.yaml"
cfg = yaml.safe_load((H / EXP).read_text())
os.environ.update({
    "H": str(H), "EXP_CFG": str(H / EXP), "EXP_NAME": cfg["experiment"],
    "VILLA_SHA": cfg["upstream"]["villa_commit"], "PY_VER": cfg["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg["upstream"]["uv_sync_args"]),
    "VES": "/kaggle/working/villa/vesuvius", "VILLA": "/kaggle/working/villa",
    "DATA": "/kaggle/working/train_data", "OUT": f"/kaggle/working/outputs/{cfg['experiment']}",
    "TIMEOUT": str(cfg["timeout_seconds"]),
})
print({k: os.environ[k] for k in ["EXP_NAME", "VILLA_SHA", "TIMEOUT"]})

## 1. Upstream code (villa, pinned commit) + environment

In [ ]:
%%bash
set -e
git init -q /kaggle/working/villa && cd /kaggle/working/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -5
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"
uv run --no-sync python -c "import accelerate; print('accelerate', accelerate.__version__)"

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing here runs `vesuvius.accept_terms`.
If a later cell fails with an error about terms/agreement, **stop and ask the owner**.

## 3. Sparse training data for 2 small segments (labels byte-exact; volume pooled to 21 slices of 9.6 um)

In [ ]:
%%bash
set -eo pipefail
cd "$VES" && uv run --no-sync python "$H/scripts/exp006_prepare_train_data.py" --config "$EXP_CFG" --out "$DATA" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -60
du -sh "$DATA"

## 4. Training configs derived from the official recipe (validated with villa's TrainingConfig)

In [ ]:
%%bash
set -eo pipefail
mkdir -p "$OUT/runs" "$OUT/logs"
cd "$VES" && uv run --no-sync python "$H/scripts/exp006_make_configs.py" --config "$EXP_CFG" --villa "$VILLA" --data "$DATA" --out "$OUT" | tail -40

## 5. Speed benchmarks (150 steps each, 30 warm-up steps excluded). A failing run (e.g. out of memory) is recorded, not fatal.

In [ ]:
%%bash
cd "$VES"
python3 -c "
import json
for r in json.load(open('$OUT/runs.json')):
    print(r['id'], r['gpus'], r['config'], r['kind'])" > "$OUT/runs.txt"
while read -r id gpus cfgp kind; do
  [ "$kind" = benchmark ] || continue
  echo "=== $id ($gpus GPU) $(date +%T) ==="
  if [ "$gpus" = 1 ]; then
    CUDA_VISIBLE_DEVICES=0 timeout "$TIMEOUT" uv run --no-sync python -m vesuvius.ink_detection.training.train "$cfgp" > "$OUT/logs/$id.log" 2>&1
  else
    timeout "$TIMEOUT" uv run --no-sync accelerate launch --num_processes "$gpus" --module vesuvius.ink_detection.training.train "$cfgp" > "$OUT/logs/$id.log" 2>&1
  fi
  echo "exit status $? | benchmark json: $(ls "$OUT/runs/${id}_benchmark.json" 2>/dev/null || echo MISSING)"
done < "$OUT/runs.txt"

## 6. Functional run: 300 steps with validation and a checkpoint (shows that the full training path works)

In [ ]:
%%bash
cd "$VES"
while read -r id gpus cfgp kind; do
  [ "$kind" = functional ] || continue
  echo "=== $id ($gpus GPU) $(date +%T) ==="
  CUDA_VISIBLE_DEVICES=0 timeout "$TIMEOUT" uv run --no-sync python -m vesuvius.ink_detection.training.train "$cfgp" > "$OUT/logs/$id.log" 2>&1
  echo "exit status $?"; ls -la "$OUT/runs/$id" | head -20
done < "$OUT/runs.txt"

## 7. Summary (numbers only)

In [ ]:
%%bash
cd "$VES" && uv run --no-sync python "$H/scripts/exp006_summarize.py" --out "$OUT"
cp "$DATA/train_data.json" "$OUT/train_data.json"
for f in "$OUT"/logs/*.log; do echo "--- $(basename $f) (last 4 lines)"; tail -n 4 "$f" | cut -c1-240; done
ls -la "$OUT" "$OUT/runs"

## 8. Bring results back
Files in `/kaggle/working/outputs/exp006_train_benchmark/`: `summary.json`, `train_data.json`, `runs/*_benchmark.json`
(and, if short, the last lines of the logs printed above). Verify each against `sha256sum` / `wc -c` run in a cell.
Do **not** download checkpoints or previews.